In [ ]:
import csv # for reading CSV files
import os # for file and directory operations
import pandas as pd # for data manipulation
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # for tokenization and model loading
from sklearn.model_selection import train_test_split # for splitting the dataset

directory = r"../data/raw/ie/" # directory containing the CSV files

cleaned_dfs = [] # list to hold data from all CSV files

# check if directory exists
if not os.path.exists(directory):
    print(f"Directory '{directory}' does not exist.")
    exit()
else:
    # if the directory exists, process each CSV file
    print(f"Processing files in directory: {directory}")
    for filename in os.listdir(directory):
        if filename.endswith(".csv"):
            file_path = os.path.join(directory, filename)
            try:
                # read the CSV file
                df = pd.read_csv(file_path)
                # remove all columns except for Content and Summary
                df = df[['Content', 'Summary']]
                # drop empty rows
                df.dropna(how='any', inplace=True)
                # append the DataFrame to the list
                cleaned_dfs.append(df)
            except Exception as e:
                print(f"Error while processing file {filename}: {e}")

print("All CSV files processed.")

# concatenate all DFs into a single dataframe
if cleaned_dfs:
    combined_df = pd.concat(cleaned_dfs, ignore_index=True)
else: # odd scenario where no CSV files were found
    print("No valid CSV files found.")
    exit()

# Clean the Text in the merged DataFrame
# 1. Remove leading and trailing whitespace
combined_df['Content'] = combined_df['Content'].str.strip()
combined_df['Summary'] = combined_df['Summary'].str.strip()
# 2. Remove any unnecessary characters
combined_df['Content'] = combined_df['Content'].str.replace(r'\n+', '', regex=True) # new lines
combined_df['Summary'] = combined_df['Summary'].str.replace(r'\n+', '', regex=True) # new lines
combined_df['Content'] = combined_df['Content'].str.replace(r'<.*?>', '', regex=True) # HTML tags
combined_df['Summary'] = combined_df['Summary'].str.replace(r'<.*?>', '', regex=True) # HTML tags
combined_df['Content'] = combined_df['Content'].str.replace(r'\xa0', '', regex=True) # non-breaking spaces
combined_df['Summary'] = combined_df['Summary'].str.replace(r'\xa0', '', regex=True) # non-breaking spaces
combined_df['Content'] = combined_df['Content'].str.replace(r'http\S+', '', regex=True) # links
combined_df['Summary'] = combined_df['Summary'].str.replace(r'http\S+', '', regex=True) # links
# lowercase the text
combined_df['Content'] = combined_df['Content'].str.lower()
combined_df['Summary'] = combined_df['Summary'].str.lower()
# remove duplicate rows
combined_df.drop_duplicates(inplace=True)

# remove rows where Content or Summary is empty
combined_df = combined_df[(combined_df['Content'].str.strip() != '') & (combined_df['Summary'].str.strip() != '')]

print(f"Combined DataFrame Head:\n{combined_df.head()}")

# Chunking the DataFrame into smaller parts
chunk_size_content = 512 # size of each chunk

# tokenize the input text and check token length
tokenizer = AutoTokenizer.from_pretrained("../models/bart-base")

chunked_data = []

# iterate through the rows of the DataFrame
for _, row in combined_df.iterrows():
    content = row['Content']
    summary = row['Summary']

    content_tokens = tokenizer.encode(content, truncation=False)
    token_length = len(content_tokens)

    print("Token Length: ", token_length)

    # if the content is too long, split it into chunks
    if token_length > chunk_size_content:
        # split the content into chunks
        content_chunks = [content_tokens[i:i+chunk_size_content] for i in range(0, len(content_tokens), chunk_size_content)]

        # create a new row for each chunk
        for content_chunk in content_chunks:
            chunked_data.append({
                'Content': tokenizer.decode(content_chunk, skip_special_tokens=True),
                'Summary': summary  # keep the original summary
            })
    else:
        chunked_data.append({
            'Content': content,
            'Summary': summary
        })

# create a new DataFrame from the chunked data
chunked_df = pd.DataFrame(chunked_data)

# 70-30 split of the chunked DataFrame
train_df, test_df = train_test_split(chunked_df, test_size=0.3, random_state=42)

os.makedirs("../data/cleaned", exist_ok=True)
train_df.to_csv("../data/cleaned/train.csv", index=False)
test_df.to_csv("../data/cleaned/test.csv", index=False)



print("Done. Train and test CSVs saved.")

Processing files in directory: ../data/raw/ie/
All CSV files processed.
Combined DataFrame Head:
                                             Content  \
0  with the delhi assembly elections fast approac...   
1  the shiromani gurdwara parbandhak committee sg...   
2  the sajad gani lone-led jammu and kashmir peop...   
3  a day after uttar pradesh minister ashish pate...   
4  with senior congress leader and the partys mah...   

                                             Summary  
0  if caste impacts peoples lives from birth to d...  
1  this is perhaps for the first time that the si...  
2  while the police clearance process was there e...  
3  a day after uttar pradesh minister ashish pate...  
4  at present he is only a permanent invitee to t...  
Token Length:  893
Token Length:  1045
Token Length:  750
Token Length:  675
Token Length:  745
Token Length:  1554
Token Length:  907
Token Length:  1265
Token Length:  1134
Token Length:  1993
Token Length:  675
Token Length:  994
Tok